# PoliMillionaire — Maths: 5-Arm Comparison

This notebook puts **five maths-question approaches side by side** so they can be run and compared
**both offline and online**, without altering any of the original implementations.

**Note**: In this notebook although the functions and implementation details for gemma-4 model are provided, in the final results and comparison it's not included. Experiment with gemma-4 model couldn't succeed due to being consistenly aborted. But we decided to keep it here for the sake of completeness and showed that we also tried with that model.

The arms (each kept verbatim from its source notebook):

| # | Arm | Source notebook | Model | What it does |
|---|-----|-----------------|-------|--------------|
| 1 | **Sandbox tool** | `nlp_maths` | Qwen2.5-14B | model writes a short Python snippet -> we `exec` it in a sandbox -> model maps the printed result to an option (`ask_llm_math_tool`). |
| 2 | **Sandbox tool + gate** | `nlp_maths` + gating idea from `…v2-fixed` | Qwen2.5-14B | same sandbox tool, but a cheap **gate** first decides whether code is even needed; conceptual questions skip straight to the baseline answer path. |
| 3 | **CoT** | `nlp-project-cot-tool-baseline` | Qwen2.5-14B | `ask_llm(strategy=P_MATHS_COT, use_tools=False)` — brief reasoning then `Answer: N`. |
| 4 | **CoT** | `nlp-project-cot-tool-baseline` | Gemma-4-E4B | same CoT strategy, Gemma-4. |
| 5 | **Baseline (direct)** | `nlp-project-cot-tool-baseline` | Qwen2.5-14B | `ask_llm(strategy=P0_BASELINE)` — single direct-answer pass. |

Switching models between arms is done with `load_model_globally(...)`; the comparison runners at the bottom
handle the swap for you so arms on different models are evaluated correctly.

# Libraries

In [ ]:
!pip -q install transformers accelerate bitsandbytes sentencepiece
!pip -q install ddgs sentence-transformers
!pip -q install trafilatura googlenewsdecoder rank-bm25
!pip -q install mistral-common --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 93.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 2

In [ ]:
# this might be needed depending on chosen model and environment
# uncomment if you see relevant error and restart kernel after executing this cell (crucial)
!pip -q install "transformers[serving] @ git+https://github.com/huggingface/transformers.git@main"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import time
import torch
import json
import hashlib
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    AutoProcessor,
    Gemma3ForConditionalGeneration,
    Mistral3ForConditionalGeneration
)
from mistral_common.protocol.instruct.request import ChatCompletionRequest
from mistral_common.tokens.tokenizers.mistral import MistralTokenizer
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import sys
import os
import re
from tqdm import tqdm
from dataclasses import dataclass
from typing import Callable, Union, Optional, List
from collections import Counter
import shutil
import requests
from urllib.parse import quote
from dataclasses import replace

# --- added for CoT tool definitions ---
import math
import sympy as sp
from sympy import symbols

# --- added for the nlp_maths Python sandbox arm ---
import statistics, itertools, signal, contextlib, io
from scipy import stats as scipy_stats

In [ ]:
# Fetch the secret from Kaggle's vault
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
# Log in to Hugging Face
login(token=hf_token)

## PoliMillionaire client

In [ ]:
# polimillionaire client
package_parent_dir = '/kaggle/input/datasets/isemzden/millionaire-client'
if package_parent_dir not in sys.path:
    sys.path.append(package_parent_dir)
print(sys.path)

from millionaire_client import MillionaireClient, AuthenticationError
pwd = user_secrets.get_secret("poli-millionaire")

API_URL = "http://131.175.15.22:51111/"
username = "cisemozden"
password = pwd

client = MillionaireClient(API_URL)
try:
    user = client.login(username, password)
    print(f"\nWelcome, {user.username}! (Role: {user.role})")
except AuthenticationError as e:
    print(f"Login failed: {e}")

['/kaggle/working', '/kaggle/lib/kagglegym', '/kaggle/lib', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython', '/kaggle/input/datasets/isemzden/millionaire-client']

Welcome, cisemozden! (Role: student)


# Model Configuration

`MODEL_NAME` is the *currently loaded* model. The comparison runners at the bottom call
`load_model_globally(...)` to swap between Qwen (arms 1, 2, 3, 5) and Gemma-4 (arm 4) as needed, so you do
not have to set this by hand for the full comparison — but it controls which model the single-arm sanity
cells use.

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct"

# Models that work for this project without fine-tuning (★ = best tested)
# Qwen:    "Qwen/Qwen2.5-7B-Instruct", "Qwen/Qwen2.5-14B-Instruct" ★, "Qwen/Qwen3-4B-Instruct-2507"
# Gemma:   "google/gemma-2-9b-it", "google/gemma-3-4b-it", "google/gemma-3-12b-it", "google/gemma-4-E4B-it" ★
# Mistral: "mistralai/Mistral-7B-Instruct-v0.3", "mistralai/Mistral-Small-3.2-24B-Instruct-2506" ★

GENERATION_CONFIG = {
    "max_new_tokens": 8,
    "do_sample": False,
    "use_cache": True,
    "output_scores": True,
    "return_dict_in_generate": True,
}

USE_QUANTIZATION = True            # set to true if model very large

# Latency we consider "risky" given the 30s server timeout
LATENCY_DANGER_THRESHOLD_S = 20.0

# Where to store / read the offline question bank
QUESTION_BANK_PATH = "/kaggle/working/question_bank.json"
QUESTION_BANK_PATH_EVAL = "/kaggle/input/datasets/isemzden/millionaire-client/question_bank.json"

In [ ]:
# model family helpers

def is_gemma3(model_name):

    model_name = model_name.lower()

    return "gemma-3" in model_name


def is_gemma2(model_name):

    model_name = model_name.lower()

    return "gemma-2" in model_name


def is_gemma4(model_name):

    model_name = model_name.lower()

    return "gemma-4" in model_name


def is_mistral3(model_name):

    model_name = model_name.lower()

    return "mistral-small-3" in model_name or "magistral" in model_name

In [ ]:
# quantization (4-bit) and model args
if USE_QUANTIZATION:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    model_kwargs = {
        "quantization_config": bnb_config,
        "device_map": "auto",
        "dtype": torch.float16,
    }
else:
    model_kwargs = {
        "device_map": "auto",
        "dtype": torch.float16,
    }

In [ ]:
# load/swap model (and tokenizer where needed)

# paths for models mounted via Kaggle Models / Datasets
# set to None (or omit) for models we want to always fetch from HF hub
KAGGLE_MODEL_PATHS = {
    "Qwen/Qwen2.5-14B-Instruct":
        "/kaggle/input/models/qwen-lm/qwen2.5/transformers/14b-instruct/1",
    "google/gemma-4-E4B-it":
        "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1",
}


def _resolve_model_path(model_name):
    """
    Return (path_or_name, source_label).
    Prefers a Kaggle-mounted local copy when present; otherwise returns the
    HF hub name so transformers will download (and cache) it.
    """
    local = KAGGLE_MODEL_PATHS.get(model_name)
    if local and Path(local).exists():
        return local, "kaggle"
    return model_name, "hub"


def load_model_globally(model_name):
    """
    Load (or swap to) a model.
    - Frees GPU memory and globals of any currently loaded model.
    - Uses /kaggle/input/... when available, else HF hub.
    - HF hub uses its on-disk cache automatically, so a previously-downloaded
      model is reloaded for free on a later swap-back (no re-download).
    """
    global MODEL_NAME, model, tokenizer, processor, mistral_tokenizer
    import gc

    # Free GPU memory + globals of whatever was loaded before
    for var in ("model", "tokenizer", "processor", "mistral_tokenizer"):
        if var in globals() and globals()[var] is not None:
            globals()[var] = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    MODEL_NAME = model_name
    source_path, source = _resolve_model_path(model_name)
    print(f"Loading {model_name}  (source: {source}, path: {source_path})")

    if is_mistral3(MODEL_NAME):
        # Mistral's chat tokenizer is separate. If we have a local copy,
        # try loading the tokenizer file from it; else fall back to hub.
        mistral_tokenizer = None
        if source == "kaggle":
            for tk_name in ("tekken.json", "tokenizer.json"):
                tk_path = Path(source_path) / tk_name
                if tk_path.exists():
                    mistral_tokenizer = MistralTokenizer.from_file(str(tk_path))
                    break
        if mistral_tokenizer is None:
            mistral_tokenizer = MistralTokenizer.from_hf_hub(MODEL_NAME)

        model = Mistral3ForConditionalGeneration.from_pretrained(
            source_path, **model_kwargs
        ).eval()
        tokenizer = None
        processor = None

    elif is_gemma3(MODEL_NAME):
        processor = AutoProcessor.from_pretrained(source_path)
        model = Gemma3ForConditionalGeneration.from_pretrained(
            source_path, **model_kwargs
        ).eval()
        tokenizer = None
        mistral_tokenizer = None

    elif is_gemma4(MODEL_NAME):
        # gemma-4 uses a processor (not a bare tokenizer); the generation
        # code routes through processor.apply_chat_template for this family.
        processor = AutoProcessor.from_pretrained(source_path)
        model = AutoModelForCausalLM.from_pretrained(
            source_path, trust_remote_code=True, **model_kwargs
        ).eval()
        tokenizer = None
        mistral_tokenizer = None

    else:
        tokenizer = AutoTokenizer.from_pretrained(source_path, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            source_path, trust_remote_code=True, **model_kwargs
        )
        processor = None
        mistral_tokenizer = None

    print(f"  loaded: {model_name}")
    return model

In [ ]:
model = load_model_globally(MODEL_NAME)

Loading Qwen/Qwen2.5-14B-Instruct  (source: hub, path: Qwen/Qwen2.5-14B-Instruct)


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

  loaded: Qwen/Qwen2.5-14B-Instruct


# Prompt Engineering (shared)

In [ ]:
from dataclasses import dataclass, field
from typing import Union, Callable, Optional, Any

@dataclass
class PromptStrategy:
    """
    Bundles everything needed to evaluate a prompting approach.
    """
    name: str
    system_prompt: Union[str, Callable[[Optional[int]], str]]
    user_builder: Callable[[str, List[str], Optional[int]], str]
    parser: Callable[[str, dict], Optional[int]]
    max_new_tokens: int = 8
    n_samples: int = 1
    do_sample: bool = False
    temperature: float = 0.0
    use_rag: bool = False
    tools: List[Any] = field(default_factory=list)


    def get_system(self, competition_id=None):
        if callable(self.system_prompt):
            return self.system_prompt(competition_id)
        return self.system_prompt


# Parsers

def parse_direct_digit(raw_output, confidence):
    """
    For direct-answer prompts. Regex for first 1-4, fallback to logit argmax.
    """
    m = re.search(r"[1-4]", raw_output.strip())
    if m:
        return int(m.group())
    if confidence:
        return max(confidence, key=confidence.get)
    return None


def parse_answer_line(raw_output, confidence):
    """
    For CoT-style outputs. Prefers explicit 'Answer: N' markers.
    """
    text = raw_output.strip()
    explicit_patterns = [
        r"(?:final\s+answer|the\s+answer\s+is|answer)\s*[:\-]?\s*([1-4])",
        r"(?:i\s+(?:choose|pick|select))\s+(?:option\s+)?([1-4])",
        r"option\s+([1-4])",
    ]
    for pat in explicit_patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            return int(m.group(1))
    # Bare digit at end of text
    m = re.search(r"([1-4])\s*[\.\)]?\s*$", text)
    if m:
        return int(m.group(1))
    # Last digit anywhere
    digits = re.findall(r"[1-4]", text)
    if digits:
        return int(digits[-1])
    if confidence:
        return max(confidence, key=confidence.get)
    return None


def parse_eliminate(raw_output, confidence):
    """
    For P5 elimination format: 'N: yes/no'. Falls through to parse_answer_line.
    """
    text = raw_output.strip()
    # Explicit Answer: line wins
    m = re.search(r"answer\s*[:\-]?\s*([1-4])", text, re.IGNORECASE)
    if m:
        return int(m.group(1))
    # Affirmative marker on a numbered option
    positive = r"(?:yes|correct|possible|right|true|likely|plausible)"
    m = re.search(rf"([1-4])\s*[:.\-]\s*{positive}\b", text, re.IGNORECASE)
    if m:
        return int(m.group(1))
    return parse_answer_line(text, confidence)


# User-prompt builders

def _user_builder_basic(question, options, competition_id=None):
    return (
        f"Question:\n{question}\n\n"
        f"Options:\n"
        f"1. {options[0]}\n"
        f"2. {options[1]}\n"
        f"3. {options[2]}\n"
        f"4. {options[3]}\n\n"
        f"Answer:"
    )


# P0: direct-answer baseline

P0_BASELINE = PromptStrategy(
    name="P0_baseline",
    system_prompt=(
        "You are a multiple-choice trivia expert.\n\n"
        "You will receive one question and four numbered options.\n"
        "Choose the single most likely correct option.\n\n"
        "Output format:\n"
        "- Reply with ONLY one digit: 1, 2, 3, or 4.\n"
        "- No words, no punctuation, no explanation.\n"
        "- If unsure, still pick your best guess. You cannot skip."
    ),
    user_builder=_user_builder_basic,
    parser=parse_direct_digit,
    max_new_tokens=512,
)

P0_BASELINE_try = PromptStrategy(
    name="P0_baseline",
    system_prompt=(
        "You are a multiple-choice trivia expert.\n\n"
        "You will receive one question and four numbered options.\n"
        "Choose the single most likely correct option.\n\n"
        "Output format:\n"
        "- First briefly explain your reasoning along with the formulas.\n"
        "- Then reply with ONLY one digit: 1, 2, 3, or 4.\n"
        "- If unsure, still pick your best guess. You cannot skip."
    ),
    user_builder=_user_builder_basic,
    parser=parse_direct_digit,
    max_new_tokens=512,
)


# P1: few-shot

P1_FEWSHOT = PromptStrategy(
    name="P1_fewshot",
    system_prompt=(
        "You are a multiple-choice trivia expert.\n"
        "Reply with ONLY one digit (1, 2, 3, or 4). No explanation.\n\n"
        "Examples:\n\n"
        "Question: What is the chemical symbol for gold?\n"
        "Options:\n1. Au\n2. Ag\n3. Go\n4. Gd\nAnswer: 1\n\n"
        "Question: What is 12 multiplied by 7?\n"
        "Options:\n1. 72\n2. 84\n3. 96\n4. 108\nAnswer: 2\n\n"
        "Question: What is the largest planet in our solar system?\n"
        "Options:\n1. Earth\n2. Mars\n3. Jupiter\n4. Saturn\nAnswer: 3\n\n"
        "Question: In which year did World War II end?\n"
        "Options:\n1. 1943\n2. 1944\n3. 1946\n4. 1945\nAnswer: 4\n"
    ),
    user_builder=_user_builder_basic,
    parser=parse_direct_digit,
    max_new_tokens=8,
)


# P2: chain-of-thought

P2_COT = PromptStrategy(
    name="P2_cot",
    system_prompt=(
        "You are a multiple-choice trivia expert.\n\n"
        "Think briefly about which option is correct — at most 3 short sentences.\n"
        "Then on the final line write exactly:\n"
        "Answer: N\n"
        "where N is 1, 2, 3, or 4."
    ),
    user_builder=_user_builder_basic,
    parser=parse_answer_line,
    max_new_tokens=128,
)


# P3: CoT + self-consistency

P3_SELF_CONSISTENCY = PromptStrategy(
    name="P3_self_consistency",
    system_prompt=P2_COT.system_prompt,
    user_builder=_user_builder_basic,
    parser=parse_answer_line,
    max_new_tokens=128,
    n_samples=3,           # increase to 5 if accuracy boost justifies the latency
    do_sample=True,
    temperature=0.7,
)


# P4: per-competition specialization

_P4_SYSTEMS = {
    0: (  # Entertainment
        "You are an expert on entertainment trivia (movies, music, TV, "
        "games, books, celebrities). Recall the precise fact and reply "
        "with ONLY one digit (1, 2, 3, or 4)."
    ),
    1: (  # Ancient History and Politics
        "You are an expert on ancient history and politics. Consider "
        "dates, historical context, and political systems carefully. "
        "Reply with ONLY one digit (1, 2, 3, or 4)."
    ),
    2: (  # Science and Nature
        "You are an expert on science and nature. Apply established "
        "scientific principles and verified knowledge. Reply with ONLY "
        "one digit (1, 2, 3, or 4)."
    ),
    3: (  # Maths
        "You are an expert at multiple-choice mathematics problems.\n"
        "Work the problem step by step in 2-4 short lines.\n"
        "Then on the final line write exactly:\nAnswer: N\n"
        "where N is 1, 2, 3, or 4."
    ),
}

def _p4_system(competition_id):
    return _P4_SYSTEMS.get(competition_id, P0_BASELINE.system_prompt)

P4_SPECIALIZED = PromptStrategy(
    name="P4_specialized",
    system_prompt=_p4_system,
    user_builder=_user_builder_basic,
    parser=parse_answer_line,
    max_new_tokens=128,
)


# P5: eliminate impossible options

P5_ELIMINATE = PromptStrategy(
    name="P5_eliminate",
    system_prompt=(
        "You are a multiple-choice trivia expert.\n\n"
        "For each option, very briefly note whether it could be correct.\n"
        "Use the format 'N: yes/no — short reason' on one line per option.\n"
        "Then on the final line write exactly:\n"
        "Answer: N\n"
        "where N is 1, 2, 3, or 4.\n\n"
        "Example:\n"
        "1: no — wrong century\n"
        "2: yes — matches the description\n"
        "3: no — refers to a different event\n"
        "4: no — geographic mismatch\n"
        "Answer: 2"
    ),
    user_builder=_user_builder_basic,
    parser=parse_eliminate,
    max_new_tokens=200,
)


# LEGACY: kept for reference only. Do NOT use as a planner. The planner role is P_PLANNER (below).
P6_USE_TOOL = PromptStrategy(
    name="P6_use_tool",
    system_prompt=(
        "You are a multiple-choice maths expert.\n\n"
        "You will receive a question, four numbered options, and possibly a "
        "block of calculated results from tools. If tool results are provided, "
        "TRUST them and do NOT recompute by hand.\n\n"
        "Match the tool result to the correct option, then on the final line "
        "write exactly:\nAnswer: N\n"
        "where N is 1, 2, 3, or 4."
    ),
    user_builder=_user_builder_basic,
    parser=parse_answer_line,
    max_new_tokens=512,
)

ALL_STRATEGIES = [P0_BASELINE, P1_FEWSHOT, P2_COT, P3_SELF_CONSISTENCY, P4_SPECIALIZED, P5_ELIMINATE, P6_USE_TOOL]
# RAG-augmented strategy variants — identical to the originals but with use_rag=True.
P0_RAG = replace(P0_BASELINE, name="P0_baseline_rag", use_rag=True)
P2_RAG = replace(P2_COT,      name="P2_cot_rag",      use_rag=True)
P4_RAG = replace(P4_SPECIALIZED, name="P4_specialized_rag", use_rag=True)
RAG_STRATEGIES = [P0_RAG, P2_RAG, P4_RAG]

# (A) A maths-only CoT system prompt
# This is the prompt that was actually doing the work:
# letting the model reason, then forcing a single "Answer: N" line. We cap
# the tokens tightly (latency control) instead of the original 512.
_MATHS_COT_SYSTEM = (
    "You are an expert at multiple-choice mathematics.\n"
    "Reason in AT MOST 3 very short lines. Be terse.\n"
    "If a tool result block is provided, TRUST it and do not recompute.\n"
    "CRITICAL: your reasoning budget is limited, so you MUST commit to an "
    "answer quickly. As soon as you can, and ALWAYS before you run out of "
    "room, write the final line exactly as:\n"
    "Answer: N\n"
    "where N is 1, 2, 3, or 4. The 'Answer:' line is mandatory; never end "
    "without it. Output nothing after it."
)

P_MATHS_COT = PromptStrategy(
    name="P_maths_cot",
    system_prompt=_MATHS_COT_SYSTEM,
    user_builder=_user_builder_basic,
    parser=parse_answer_line,
    max_new_tokens=160,        # tightened from 256: forces an early commit and
                               # keeps the hard cases under the 30s wall
)

# (B) A dedicated PLANNING system prompt --------------------------------
_PLANNER_SYSTEM = (
    "You decide whether a calculator tool is needed to answer a maths "
    "question. If a numeric computation (arithmetic, combinatorics, number "
    "theory, growth) is required, emit the appropriate tool call. If the "
    "question is conceptual/theoretical and needs no computation, emit no "
    "tool call at all. Do NOT answer the question yourself in this step."
)

P_PLANNER = PromptStrategy(
    name="P_planner",
    system_prompt=_PLANNER_SYSTEM,
    user_builder=_user_builder_basic,
    parser=parse_answer_line,
    max_new_tokens=96,
)

# Shared Inference Backbone

In [ ]:
# whole message-to-llm builder

def build_messages(model_name, system_prompt, user_prompt):
    model_name = model_name.lower()

    # GEMMA 3 FORMAT
    if is_gemma3(model_name):
        messages = [
            {"role": "system", "content": [{"type": "text", "text": system_prompt,}]},
            {"role": "user", "content": [{"type": "text", "text": user_prompt,}]}
        ]
        return messages

    # GEMMA 2 FORMAT
    elif is_gemma2(model_name):
        merged_prompt = (
            system_prompt.strip()
            + "\n\n"
            + user_prompt.strip()
        )
        messages = [
            {"role": "user", "content": merged_prompt,}
        ]
        return messages

    # STANDARD CHAT TEMPLATE MODELS
    else:
        messages = [
            {"role": "system", "content": system_prompt,},
            {"role": "user", "content": user_prompt,},
        ]
        return messages

In [ ]:
# clean model output from text and logits

def extract_answer_from_text(text):
    """
    Parse the model's generated string for a 1-4 answer.
    Returns int in {1,2,3,4} or None if no digit found.
    """
    text = text.strip()
    match = re.search(r"[1-4]", text)
    return int(match.group()) if match else None


def extract_answer_from_logits(scores_step0, decode_fn, top_k=50):
    """
    Compute answer probabilities by inspecting the model's first-step logits.
    Walks the top-k most likely tokens at the first generation position and
    aggregates probability mass over those that decode to '1', '2', '3', '4'
    (stripped). This handles tokenizer quirks (leading-space tokens etc.)
    without needing per-tokenizer special cases.
    Returns:
        best_answer: int in {1,2,3,4} or None if no answer-token in top-k
        probs: dict {1: p1, 2: p2, 3: p3, 4: p4}, renormalized to sum to 1
    """
    probs = torch.softmax(scores_step0[0].float(), dim=-1)
    top_probs, top_ids = probs.topk(top_k)

    answer_probs = {1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0}
    for i in range(top_k):
        tok_id = top_ids[i].item()
        try:
            decoded = decode_fn(tok_id).strip()
        except Exception:
            continue
        if decoded in ("1", "2", "3", "4"):
            answer_probs[int(decoded)] += top_probs[i].item()

    total = sum(answer_probs.values())
    if total == 0:
        # No answer token found in the top-k; fall back to uniform
        return None, {k: 0.25 for k in (1, 2, 3, 4)}

    answer_probs = {k: v / total for k, v in answer_probs.items()}
    best = max(answer_probs, key=answer_probs.get)
    return best, answer_probs

In [ ]:
# SAFE ANSWER PARSER
def parse_answer_line_safe(raw_output, confidence):
    text = (raw_output or "").strip()

    explicit_patterns = [
        r"(?:final\s+answer|the\s+answer\s+is|answer)\s*[:\-]?\s*([1-4])",
        r"(?:i\s+(?:choose|pick|select))\s+(?:option\s+)?([1-4])",
        r"\boption\s+([1-4])\b",
    ]
    for pat in explicit_patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            return int(m.group(1))

    if confidence:
        best = max(confidence, key=confidence.get)
        if abs(confidence[best] - 0.25) > 1e-6:
            return best

    m = re.search(r"(?:^|\n)\s*([1-4])\s*[\.\)]?\s*$", text)
    if m:
        return int(m.group(1))

    return None

In [ ]:
def calculate_combinatorics(n: int, r: int, order_matters: bool):
    """
    Calculates the number of ways to choose 'r' items from 'n' items.

    Args:
        n: The total number of items to choose from.
        r: The number of items being chosen.
        order_matters: True for permutations (order matters), False for combinations (order does not matter).
    """
    if n < 0 or r < 0 or r > n:
        return {"error": "Invalid inputs. n and r must be positive, and r cannot be greater than n."}

    if order_matters:
        # Permutations: nPr = n! / (n-r)!
        result = math.perm(n, r)
        return {"operation": f"{n}P{r}", "result": result}
    else:
        # Combinations: nCr = n! / (r! * (n-r)!)
        result = math.comb(n, r)
        return {"operation": f"{n}C{r}", "result": result}

def calculate_normal_distribution(value: float = None, mean: float = None, std_dev: float = None, z_score: float = None):
    """
    Calculates statistics for a normal distribution. Provide either (value, mean, std_dev)
    to get a z-score, or (z_score) to get the cumulative probability.

    Args:
        value: The observed value (X).
        mean: The population mean (μ).
        std_dev: The population standard deviation (σ).
        z_score: The calculated z-score.
    """
    if value is not None and mean is not None and std_dev is not None:
        if std_dev <= 0:
            return {"error": "Standard deviation must be greater than zero."}
        calculated_z = (value - mean) / std_dev
        # Calculate cumulative probability using the error function
        percentile = 0.5 * (1 + math.erf(calculated_z / math.sqrt(2)))
        return {
            "z_score": round(calculated_z, 4),
            "percentile": round(percentile * 100, 2),
            "interpretation": f"The value {value} is {round(calculated_z, 2)} standard deviations from the mean."
        }

    elif z_score is not None:
        percentile = 0.5 * (1 + math.erf(z_score / math.sqrt(2)))
        return {"z_score": z_score, "percentile": round(percentile * 100, 2)}

    return {"error": "Insufficient arguments provided."}


def compute_math(expression: str):
    """
    Evaluates or simplifies a mathematical expression exactly using a symbolic
    math engine. Handles arithmetic with large numbers, fractions, powers,
    roots, logarithms, factorials, binomial coefficients, derivatives,
    integrals, summations, and limits. Returns both an exact and a decimal form.

    Use this for ANY calculation the question requires — do not compute by hand.

    Args:
        expression: A math expression in Python/SymPy syntax. Examples:
            "binomial(23, 6)"            -> combinations C(23,6)
            "factorial(5)"               -> 5!
            "(1 + sqrt(5))/2"            -> exact + decimal
            "log(2, 8)"                  -> log base 8 of 2
            "Rational(17, 40) * 160"     -> exact fraction arithmetic
            "diff(x + exp(x), x)"        -> derivative
            "integrate(x/(1 + x**3), (x, 0, oo))"  -> definite integral
            "summation(k, (k, 1, 100))"  -> sum
            "solve(x**2 + 2*x + 7 - (6*x + b), x)" (declare symbols in text)
    """
    try:
        # common single-letter symbols the model may use without declaring
        local = {s: symbols(s) for s in
                 ("x", "y", "z", "n", "k", "a", "b", "c", "r", "s", "t")}
        local.update({
            "binomial": sp.binomial, "factorial": sp.factorial,
            "sqrt": sp.sqrt, "log": sp.log, "ln": sp.log, "exp": sp.exp,
            "sin": sp.sin, "cos": sp.cos, "tan": sp.tan, "pi": sp.pi,
            "oo": sp.oo, "E": sp.E, "Rational": sp.Rational,
            "diff": sp.diff, "integrate": sp.integrate,
            "summation": sp.summation, "Sum": sp.Sum, "limit": sp.limit,
            "solve": sp.solve, "simplify": sp.simplify, "factor": sp.factor,
            "expand": sp.expand, "gcd": sp.gcd, "lcm": sp.lcm,
            "fibonacci": sp.fibonacci, "isprime": sp.isprime,
            "divisors": sp.divisors, "totient": sp.totient,
            "Abs": sp.Abs, "floor": sp.floor, "ceiling": sp.ceiling,
            "Mod": sp.Mod, "primerange": sp.primerange,
        })
        expr = eval(expression, {"__builtins__": {}}, local)  # restricted namespace
        exact = sp.simplify(expr) if not isinstance(expr, (list, tuple, dict)) else expr
        result = {"expression": expression, "exact": str(exact)}
        try:
            result["decimal"] = float(sp.N(exact))
        except Exception:
            result["decimal"] = None
        return result
    except Exception as e:
        return {"error": f"Could not evaluate '{expression}': {e}"}


def number_theory(operation: str, a: int, b: int = 0):
    """
    Integer / number-theory operations.

    Args:
        operation: One of "gcd", "lcm", "is_prime", "factorize",
            "divisors" (uses a only), "modulo", "factorial" (uses a only).
        a: Primary integer.
        b: Secondary integer (for gcd, lcm, modulo).
    """
    if operation == "gcd":
        return {"result": math.gcd(a, b)}
    if operation == "lcm":
        return {"result": math.lcm(a, b)}
    if operation == "is_prime":
        return {"is_prime": bool(sp.isprime(a))}
    if operation == "factorize":
        return {"factorization": {int(p): int(e) for p, e in sp.factorint(a).items()}}
    if operation == "divisors":
        d = [int(x) for x in sp.divisors(a)]
        return {"divisors": d, "count": len(d)}
    if operation == "modulo":
        if b == 0:
            return {"error": "Cannot mod by zero."}
        return {"result": a % b}
    if operation == "factorial":
        if a < 0:
            return {"error": "Undefined for negatives."}
        return {"result": math.factorial(a)}
    return {"error": f"Unknown operation '{operation}'."}


def compound_growth(principal: float = None, rate_percent: float = None,
                     years: float = None, final_amount: float = None):
    """
    Solves compound-growth problems final = principal * (1 + rate/100)**years.
    Provide exactly three of the four arguments; the tool solves for the fourth.

    Args:
        principal: Starting amount.
        rate_percent: Growth rate per period, in percent.
        years: Number of periods.
        final_amount: Ending amount.
    """
    given = [principal, rate_percent, years, final_amount]
    if sum(v is not None for v in given) != 3:
        return {"error": "Provide exactly three of four arguments."}
    try:
        if final_amount is None:
            return {"final_amount": round(principal * (1 + rate_percent/100)**years, 6)}
        if principal is None:
            return {"principal": round(final_amount / (1 + rate_percent/100)**years, 6)}
        if years is None:
            yrs = math.log(final_amount/principal) / math.log(1 + rate_percent/100)
            return {"years": round(yrs, 6)}
        if rate_percent is None:
            rate = ((final_amount/principal)**(1/years) - 1) * 100
            return {"rate_percent": round(rate, 6)}
    except Exception as e:
        return {"error": str(e)}


maths_tools = [
    compute_math,
    calculate_combinatorics,
    number_theory,
    compound_growth,
    calculate_normal_distribution
]

tools = maths_tools

In [ ]:
# answer functions

def _decode_fn_for_model():
    """
    Return a unified decode_fn(tok_id) -> str for the active model family.
    """
    if is_mistral3(MODEL_NAME):
        return lambda tid: mistral_tokenizer.decode([tid])
    elif is_gemma3(MODEL_NAME) or is_gemma4(MODEL_NAME):
        # both gemma families use a processor (tokenizer is None)
        return lambda tid: processor.tokenizer.decode([tid])
    else:
        return lambda tid: tokenizer.decode([tid])


def extract_tool_calls(text):
    """Parse Gemma-4 native tool-call syntax: <|tool_call>call:name{k:v}<tool_call|>
    Unchanged from the official reference (matches the model's trained format)."""
    def cast(v):
        try: return int(v)
        except:
            try: return float(v)
            except: return {'true': True, 'false': False}.get(v.lower(), v.strip("'\""))
    pattern = r"(?:<\|tool_call>)?call:(\w+)\{(.*?)\}(?:<tool_call\|>)?"
    return [{
        "name": name,
        "arguments": {
            k: cast((v1 or v2).strip())
            for k, v1, v2 in re.findall(r'(\w+):(?:<\|"\|>(.*?)<\|"\|>|([^,}]*))', args)
        }
    } for name, args in re.findall(pattern, text, re.DOTALL)]


def _generate_with_messages(model, system_prompt, user_prompt, gen_config, tools,
                            keep_special_tokens=False):
    """
    One forward pass. Returns (raw_output, latency, first-step logits, messages).

    FIX (keep_special_tokens): the tool-call delimiters <|tool_call> / <tool_call|>
    are SPECIAL tokens. The original always decoded with skip_special_tokens=True,
    which deletes those delimiters, so extract_tool_calls() could never match even
    when the model emitted a valid call. The PLANNING pass must decode with
    skip_special_tokens=False (matching the official reference); the ANSWER pass
    keeps stripping them so the parser sees clean text.
    """
    messages = build_messages(MODEL_NAME, system_prompt, user_prompt)
    skip = not keep_special_tokens

    if is_mistral3(MODEL_NAME):
        tokenized = mistral_tokenizer.encode_chat_completion(
            ChatCompletionRequest(messages=messages)
        )
        input_ids = torch.tensor([tokenized.tokens], device=model.device)
        attention_mask = torch.ones_like(input_ids)
        gen_kwargs = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "pad_token_id": 2,
            **gen_config,
        }
        input_len = len(tokenized.tokens)
        decode_seq = lambda toks: mistral_tokenizer.decode(toks.tolist()).strip()

    elif is_gemma3(MODEL_NAME):
        inputs = processor.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=True,
            return_dict=True, return_tensors="pt",
        ).to(model.device, dtype=torch.bfloat16)
        gen_kwargs = {**inputs, **gen_config}
        input_len = inputs["input_ids"].shape[-1]
        decode_seq = lambda toks: processor.decode(toks, skip_special_tokens=skip).strip()

    elif is_gemma4(MODEL_NAME):
        text = processor.apply_chat_template(messages, tools=tools, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=text, return_tensors="pt").to(model.device)
        inputs.pop("mm_token_type_ids", None)
        gen_kwargs = {**inputs, **gen_config}
        input_len = inputs["input_ids"].shape[-1]
        decode_seq = lambda toks: processor.decode(toks, skip_special_tokens=skip).strip()
    else:
        prompt_str = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt_str, return_tensors="pt").to(model.device)
        gen_kwargs = {**inputs, "pad_token_id": tokenizer.eos_token_id, **gen_config}
        input_len = inputs["input_ids"].shape[1]
        decode_seq = lambda toks: tokenizer.decode(toks, skip_special_tokens=skip).strip()

    start = time.time()
    with torch.inference_mode():
        gen_out = model.generate(**gen_kwargs)
    latency = time.time() - start

    sequences = gen_out.sequences
    generated_tokens = sequences[0][input_len:]
    raw_output = decode_seq(generated_tokens)
    scores_step0 = gen_out.scores[0] if getattr(gen_out, "scores", None) else None
    return raw_output, latency, scores_step0, messages


def _format_tool_results_text(calls, results):
    """Turn executed tool calls + results into a plain-text block we can
    splice into the user prompt (model-agnostic, no template gymnastics)."""
    lines = ["Calculated results from tools:"]
    for c, r in zip(calls, results):
        args_str = ", ".join(f"{k}={v}" for k, v in c["arguments"].items())
        lines.append(f"- {c['name']}({args_str}) -> {r['response']}")
    return "\n".join(lines)


def _gemma4_apply_and_generate(model, messages, gen_config, tools,
                               keep_special_tokens=False):
    """
    GEMMA-4 ONLY. Generate from a full `messages` list (which may include an
    assistant tool_calls turn and tool_responses), following the official
    reference: apply_chat_template(..., tools=tools) -> processor -> generate.
    Returns (raw_output, latency, scores_step0).
    """
    skip = not keep_special_tokens
    text = processor.apply_chat_template(
        messages, tools=tools, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(text=text, return_tensors="pt").to(model.device)
    inputs.pop("mm_token_type_ids", None)
    gen_kwargs = {**inputs, **gen_config}
    input_len = inputs["input_ids"].shape[-1]

    start = time.time()
    with torch.inference_mode():
        gen_out = model.generate(**gen_kwargs)
    latency = time.time() - start

    generated_tokens = gen_out.sequences[0][input_len:]
    raw_output = processor.decode(generated_tokens, skip_special_tokens=skip).strip()
    scores_step0 = gen_out.scores[0] if getattr(gen_out, "scores", None) else None
    return raw_output, latency, scores_step0


def _execute_calls(calls):
    """Run parsed tool calls against the globally-defined tool functions.
    Returns the reference-shaped results list: [{name, response}, ...]."""
    results = []
    for c in calls:
        fn = globals().get(c["name"])
        if fn is None:
            results.append({"name": c["name"], "response": {"error": "unknown tool"}})
            continue
        try:
            results.append({"name": c["name"], "response": fn(**c["arguments"])})
        except Exception as e:
            results.append({"name": c["name"], "response": {"error": str(e)}})
    return results


def _gemma4_tool_turn(messages, calls, results):
    """Append the assistant tool-call turn exactly as the reference does, so the
    answer pass sees results in the trained `tool_responses` role rather than as
    free text spliced into the user message."""
    messages = list(messages)
    messages.append({
        "role": "assistant",
        "tool_calls": [{"function": c} for c in calls],
        "tool_responses": results,
    })
    return messages



def _force_answer_followup(model, system_prompt, user_prompt, prior_text,
                           decode_fn):
    """
    Recovery pass for when the main generation truncated before emitting
    'Answer: N' (the hard questions hit the token cap mid-reasoning and would
    otherwise parse to None). We replay the question + whatever reasoning was
    produced, then ask ONLY for the digit, capped at a handful of tokens.
    Returns (answer_int_or_None, confidence_dict, latency).
    """
    followup_user = (
        user_prompt
        + "\n\nReasoning so far:\n" + (prior_text or "").strip()
        + "\n\nBased on the above, give ONLY the final choice now.\n"
          "Answer: "
    )
    cfg = {
        "use_cache": True,
        "output_scores": True,
        "return_dict_in_generate": True,
        "max_new_tokens": 8,
        "do_sample": False,
    }
    raw, lat, scores0, _ = _generate_with_messages(
        model, system_prompt, followup_user, cfg, tools=None,
        keep_special_tokens=False,
    )
    _, conf = (extract_answer_from_logits(scores0, decode_fn)
               if scores0 is not None
               else (None, {k: 0.25 for k in (1, 2, 3, 4)}))
    ans = parse_answer_line_safe(raw, conf)
    if ans is None and conf:  # last resort: take argmax logit even if ~uniform
        ans = max(conf, key=conf.get)
    return ans, conf, lat


def ask_llm(model, question, option_texts, competition_id=None, strategy=None,
            use_rag=None, tools=[], use_tools=False):
    """
    Inference wrapper.

      - use_tools=False:  a single answer pass, no tools (baseline / CoT).
      - use_tools=True:   the model decides via the P_PLANNER prompt whether a
                          tool is needed; if so it is executed and the result is
                          fed back, then the answer pass runs. This is the
                          generalizing tool route (no hand-written keyword gate).

    Returns the standard result dict.
    """
    if strategy is None:
        strategy = P0_BASELINE
    if use_rag is None:
        use_rag = strategy.use_rag

    system_prompt = strategy.get_system(competition_id)
    user_prompt = strategy.user_builder(question, option_texts, competition_id)

    total_latency = 0.0
    rag_latency = 0.0
    context = ""
    tool_calls_made, tool_results_made = [], []

    gemma4 = is_gemma4(MODEL_NAME)
    gemma4_messages = None   # set when we build a structured multi-turn list

    if use_tools and tools:
        planner_strategy = globals().get("P_PLANNER", strategy)
        planner_system = planner_strategy.get_system(competition_id)
        planner_config = {
            "use_cache": True,
            "output_scores": False,
            "return_dict_in_generate": True,
            "max_new_tokens": planner_strategy.max_new_tokens,
            "do_sample": False,
        }

        if gemma4:
            # --- official-reference flow, Gemma-4 native tool calling ---
            planner_messages = build_messages(MODEL_NAME, planner_system, user_prompt)
            raw_tool_output, step1_latency, _ = _gemma4_apply_and_generate(
                model, planner_messages, planner_config, tools=tools,
                keep_special_tokens=True,   # keep <|tool_call> delimiters
            )
            total_latency += step1_latency
            calls = extract_tool_calls(raw_tool_output)
            if calls:
                results = _execute_calls(calls)
                tool_calls_made, tool_results_made = calls, results
                gemma4_messages = _gemma4_tool_turn(planner_messages, calls, results)
        else:
            # non-gemma4: text-splice planner path
            raw_tool_output, step1_latency, _, _ = _generate_with_messages(
                model, planner_system, user_prompt, planner_config, tools=tools,
                keep_special_tokens=True,
            )
            total_latency += step1_latency
            calls = extract_tool_calls(raw_tool_output)
            if calls:
                results = _execute_calls(calls)
                tool_calls_made, tool_results_made = calls, results
                context += _format_tool_results_text(calls, results) + "\n\n---\n\n"

    if use_rag:
        rag_start = time.time()
        retrieved_docs = retriever.retrieve_and_format(question)
        rag_latency = time.time() - rag_start
        if retrieved_docs:
            context += f"References:\n{retrieved_docs}\n\n---\n\n"

    if context:
        user_prompt = context + user_prompt
    gen_config = {
        "use_cache": True,
        "output_scores": True,
        "return_dict_in_generate": True,
        "max_new_tokens": strategy.max_new_tokens,
        "do_sample": strategy.do_sample,
    }
    if strategy.do_sample:
        gen_config["temperature"] = strategy.temperature

    decode_fn = _decode_fn_for_model()
    answers, raw_outputs, confidences = [], [], []

    for _ in range(strategy.n_samples):
        if gemma4_messages is not None:
            raw_output, latency, scores_step0 = _gemma4_apply_and_generate(
                model, gemma4_messages, gen_config, tools=tools,
                keep_special_tokens=False,
            )
        else:
            raw_output, latency, scores_step0, _ = _generate_with_messages(
                model, system_prompt, user_prompt, gen_config, tools=None,
                keep_special_tokens=False,   # answer text should be clean
            )
        total_latency += latency
        raw_outputs.append(raw_output)

        _, conf = (extract_answer_from_logits(scores_step0, decode_fn)
                   if scores_step0 is not None
                   else (None, {k: 0.25 for k in (1, 2, 3, 4)}))
        ans = parse_answer_line_safe(raw_output, conf)

        if ans is None and gemma4_messages is None:
            ans, conf, fb_lat = _force_answer_followup(
                model, system_prompt, user_prompt, raw_output, decode_fn
            )
            total_latency += fb_lat

        confidences.append(conf)
        answers.append(ans)

    if strategy.n_samples > 1:
        valid = [a for a in answers if a is not None]
        if valid:
            counts = Counter(valid)
            final_answer = counts.most_common(1)[0][0]
            total = sum(counts.values())
            final_confidence = {k: counts.get(k, 0) / total for k in (1, 2, 3, 4)}
        else:
            final_answer = None
            final_confidence = {k: 0.25 for k in (1, 2, 3, 4)}
        return {
            "answer": final_answer,
            "confidence": final_confidence,
            "raw_output": raw_outputs,
            "latency": total_latency,
            "rag_latency": rag_latency,
            "samples": answers,
            "tool_calls": tool_calls_made,
            "tool_results": tool_results_made,
        }

    return {
        "answer": answers[0],
        "confidence": confidences[0],
        "raw_output": raw_outputs[0],
        "latency": total_latency,
        "rag_latency": rag_latency,
        "samples": None,
        "tool_calls": tool_calls_made,
        "tool_results": tool_results_made,
    }

# Arm 1 — `nlp_maths` Python-sandbox tool (Qwen)

The original `nlp_maths` math-tool path, dropped in **unchanged**:

- `run_python_tool` + `_SANDBOX_GLOBALS` — execute model-written code in a restricted namespace with a hard
  wall-clock timeout.
- `ask_llm_math_tool` — model writes a `python` snippet (or replies `NO_CODE`); we run it; the model then maps
  the printed result to an option. Conceptual / no-result questions fall back to the plain `ask_llm` path.

`ask_llm_math_tool` calls the shared `ask_llm` positionally exactly as written, so nothing here is modified.

In [ ]:
import re, math, statistics, itertools, signal, contextlib, io
import numpy as np
import sympy as sp
from scipy import stats as scipy_stats

# ---- the sandbox the model writes code into -------------------------------
_SANDBOX_GLOBALS = {
    "math": math, "np": np, "numpy": np, "sympy": sp, "sp": sp,
    "stats": scipy_stats, "scipy_stats": scipy_stats,
    "statistics": statistics, "itertools": itertools,
    "comb": math.comb, "perm": math.perm, "factorial": math.factorial,
    "gcd": math.gcd, "Matrix": sp.Matrix, "symbols": sp.symbols,
    "integrate": sp.integrate, "diff": sp.diff, "solve": sp.solve,
    "simplify": sp.simplify, "Rational": sp.Rational, "binom": math.comb,
    "norm": scipy_stats.norm, "t": scipy_stats.t, "chi2": scipy_stats.chi2,
    "f": scipy_stats.f, "binom_dist": scipy_stats.binom,
}

class _Timeout(Exception): pass
def _alarm(signum, frame): raise _Timeout()

def run_python_tool(code, timeout_s=8):
    """Execute model-written code in a restricted namespace; return stdout text.
    Hard wall-clock timeout so a runaway snippet can't eat the 30s budget."""
    code = re.sub(r"^```(?:python)?|```$", "", code.strip(), flags=re.M).strip()
    buf = io.StringIO()
    ns = dict(_SANDBOX_GLOBALS)
    signal.signal(signal.SIGALRM, _alarm)
    signal.alarm(int(timeout_s))
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, ns)
        out = buf.getvalue().strip()
    except _Timeout:
        out = "ERROR: timed out"
    except Exception as e:
        out = f"ERROR: {type(e).__name__}: {e}"
    finally:
        signal.alarm(0)
    return out

In [ ]:
def _extract_code(text):
    """Pull a runnable snippet from the model output: prefer a ```python``` block,
    else take everything from the first line that looks like code."""
    m = re.search(r"```(?:python)?\s*(.*?)```", text, flags=re.S)
    if m:
        return m.group(1).strip()
    lines = text.splitlines()
    for i, ln in enumerate(lines):
        if re.match(r"\s*(import |from |print\(|\w+\s*=)", ln):
            return "\n".join(lines[i:]).strip()
    return ""

def _result_is_usable(result):
    return bool(result) and not result.startswith("ERROR") and result.lower() != "none"

def ask_llm_math_tool(model, question, option_texts, competition_id=3,
                      max_code_tokens=320, verbose=False):
    """Comp-3 via a Python sandbox. Model writes a snippet -> we run it -> model
    maps the computed result to an option. Falls back to the plain answer path
    when no usable code/result is produced (e.g. conceptual questions)."""
    opts = "\n".join(f"{i+1}. {t}" for i, t in enumerate(option_texts))
    sys = ("You solve math/statistics multiple-choice questions by writing a SHORT "
           "Python program that prints the numeric or symbolic answer. Available: "
           "numpy as np, sympy as sp, scipy.stats as stats, math, statistics, itertools, "
           "comb/perm/factorial, Matrix, integrate, diff, solve, norm/t/chi2/f. "
           "If the question is purely conceptual and needs no computation, reply exactly "
           "NO_CODE. Otherwise output ONLY a ```python``` block ending in print(...).")
    user = f"Question: {question}\n\nOptions:\n{opts}"

    messages = [{"role": "system", "content": sys}, {"role": "user", "content": user}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        gen = model.generate(**inputs, max_new_tokens=max_code_tokens,
                             do_sample=False, pad_token_id=tokenizer.eos_token_id)
    raw = tokenizer.decode(gen[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    if "NO_CODE" in raw.upper():
        if verbose: print("[tool] model chose NO_CODE -> plain path")
        return ask_llm(model, question, option_texts, competition_id, None, False)

    code = _extract_code(raw)
    result = run_python_tool(code) if code else ""
    if verbose:
        print(f"--- code ---\n{code}\n--- result ---\n{result}")

    if not _result_is_usable(result):
        if verbose: print("[tool] no usable result -> plain path")
        return ask_llm(model, question, option_texts, competition_id, None, False)

    map_q = (f"{question}\n\nA Python computation produced this result: {result}\n\n"
             f"Use it to choose the correct option.")
    res = ask_llm(model, map_q, option_texts, competition_id=competition_id,
                  strategy=None, use_rag=False)
    res["tool_used"] = True
    res["tool_result"] = result
    return res

# Arm 2 — `nlp_maths` sandbox tool **+ gate** (Qwen)

Integrates the **gating idea** (a cheap pass that decides *whether* a tool is needed before paying for it) and apply it to the Python sandbox.

The original sandbox already has an implicit gate — the model may answer `NO_CODE`. Arm 2 makes that gate
**explicit and cheap**: a short, low-token decision pass (`_sandbox_gate`, capped tokens, system prompt
mirroring the v2 `TOOL_GATE_SYSTEM` wording) runs first.

- gate says **CODE** → run the full `ask_llm_math_tool` (write snippet → exec → map result).
- gate says **NONE** → skip the expensive code-writing pass and answer directly via the shared `ask_llm`
  baseline path (the v2 "gate declined → baseline" behaviour).

The underlying tool implementation (`run_python_tool`, `ask_llm_math_tool`) is **not changed** — arm 2 only
wraps it.

In [ ]:
# --- gate decision (idea adapted from the v2 TOOL_GATE_SYSTEM, applied to the sandbox) ---

_SANDBOX_GATE_SYSTEM = (
    "You decide whether a calculation is needed to answer a multiple-choice maths "
    "question.\n\n"
    "Answer CODE only if answering requires exact computation you cannot reliably do "
    "mentally: large-number arithmetic, factorials, combinations/permutations, solving "
    "equations, integrals/derivatives, gcd/lcm/prime factorization, compound interest, "
    "probability/statistics values, or unit conversion.\n\n"
    "Answer NONE for conceptual, definitional, or pure-reasoning questions (abstract "
    "algebra theory, 'which statement is true', factual recall).\n\n"
    "Reply with exactly one word: CODE or NONE."
)

def _sandbox_gate(model, question, option_texts, competition_id=3,
                  max_gate_tokens=8, verbose=False):
    """Cheap decision pass. Returns True if the sandbox tool should run.
    Uses the same tokenizer/chat path the sandbox tool uses (Qwen-family)."""
    opts = "\n".join(f"{i+1}. {t}" for i, t in enumerate(option_texts))
    user = (f"Question:\n{question}\n\nOptions:\n{opts}\n\n"
            "Decision (CODE or NONE):")
    messages = [{"role": "system", "content": _SANDBOX_GATE_SYSTEM},
                {"role": "user", "content": user}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        gen = model.generate(**inputs, max_new_tokens=max_gate_tokens,
                             do_sample=False, pad_token_id=tokenizer.eos_token_id)
    raw = tokenizer.decode(gen[0][inputs["input_ids"].shape[1]:],
                           skip_special_tokens=True).strip().upper()
    # Verdict rule: take CODE if it appears; else NONE if it appears; default to
    # CODE (fail toward computing, matching the sandbox tool's own NO_CODE fallback).
    if "CODE" in raw:
        use_code = True
    elif "NONE" in raw:
        use_code = False
    else:
        use_code = True
    if verbose:
        print(f"[sandbox-gate] verdict raw={raw!r} -> {'CODE' if use_code else 'NONE'}")
    return use_code


def ask_llm_math_tool_gated(model, question, option_texts, competition_id=3,
                            max_code_tokens=320, verbose=False):
    """Arm 2: gate first, then either the full sandbox tool or a direct answer.
    Wraps the unchanged ask_llm_math_tool / ask_llm."""
    t0 = time.time()
    use_code = _sandbox_gate(model, question, option_texts, competition_id, verbose=verbose)
    gate_latency = time.time() - t0

    if use_code:
        res = ask_llm_math_tool(model, question, option_texts,
                                competition_id=competition_id,
                                max_code_tokens=max_code_tokens, verbose=verbose)
        res["gate_decision"] = "CODE"
    else:
        if verbose:
            print("[sandbox-gate] NONE -> baseline answer path")
        # gate declined -> cheap baseline answer (mirrors the v2 'gate declined' behaviour)
        res = ask_llm(model, question, option_texts, competition_id, None, False)
        res["tool_used"] = False
        res["gate_decision"] = "NONE"

    res["latency"] = res.get("latency", 0.0) + gate_latency
    res["gate_latency"] = gate_latency
    return res

# The Five Arms — registry

Every arm is described by `(label, model_name, runner)`, where `runner(model, question, option_texts,
competition_id) -> result_dict`. The two sandbox arms (1, 2) use their dedicated runners; the three pipeline
arms (3, 4, 5) use the shared `ask_llm` with a fixed `strategy` / `use_tools`. The result dict shape
(`answer`, `confidence`, `raw_output`, `latency`, …) is common to all of them.

In [ ]:
from functools import partial

# Pipeline-arm runners: thin closures over the shared ask_llm (no implementation change).
def _runner_cot(model, q, opts, comp):
    return ask_llm(model, q, opts, competition_id=comp, strategy=P_MATHS_COT, use_rag=False)

def _runner_baseline(model, q, opts, comp):
    return ask_llm(model, q, opts, competition_id=comp, strategy=P0_BASELINE, use_rag=False)

def _runner_sandbox(model, q, opts, comp):
    return ask_llm_math_tool(model, q, opts, competition_id=comp)

def _runner_sandbox_gated(model, q, opts, comp):
    return ask_llm_math_tool_gated(model, q, opts, competition_id=comp)

# label -> (model_name, runner)
ARMS = {
    "1_sandbox_qwen":        ("Qwen/Qwen2.5-14B-Instruct", _runner_sandbox),
    "2_sandbox_gated_qwen":  ("Qwen/Qwen2.5-14B-Instruct", _runner_sandbox_gated),
    "3_cot_qwen":            ("Qwen/Qwen2.5-14B-Instruct", _runner_cot),
    "4_cot_gemma":           ("google/gemma-4-E4B-it",      _runner_cot),
    "5_baseline_qwen":       ("Qwen/Qwen2.5-14B-Instruct", _runner_baseline),
}

# Group by model so we load each model once and run all its arms before swapping.
def arms_by_model(selected=None):
    selected = selected or list(ARMS)
    by_model = {}
    for label in selected:
        mname, runner = ARMS[label]
        by_model.setdefault(mname, []).append((label, runner))
    return by_model

print("Registered arms:")
for k, (m, _) in ARMS.items():
    print(f"  {k:24s} -> {m}")

Registered arms:
  1_sandbox_qwen           -> Qwen/Qwen2.5-14B-Instruct
  2_sandbox_gated_qwen     -> Qwen/Qwen2.5-14B-Instruct
  3_cot_qwen               -> Qwen/Qwen2.5-14B-Instruct
  4_cot_gemma              -> google/gemma-4-E4B-it
  5_baseline_qwen          -> Qwen/Qwen2.5-14B-Instruct


# Gameplay

`play_leaderboard` drives the live game through the shared `ask_llm`; the comparison runners below use either
this (for the pipeline arms) or arm-specific runners (for the sandbox arms).

In [ ]:
def play_leaderboard(model, game, question_bank=None, eval_log=None,
                     verbose=True, strategy=None, use_rag=None,
                     use_tools=False, tools=None):
    """
    Play one game against the live API. NEVER consults the question_bank.
    use_rag: optional override; if None, uses strategy.use_rag.
    """
    if strategy is None:
        strategy = P0_BASELINE

    while game.in_progress:
        question = game.current_question
        if not question:
            if verbose:
                print("No question available. Game may have ended.")
            break

        option_texts = [opt.text for opt in question.options]
        comp_id = game.state.competition.id
        level = game.current_level

        if verbose:
            print(f"\n--- Level {level} ---")
            print(f"Q: {question.text}")
            for i, opt in enumerate(question.options):
                print(f"  [{i+1}] {opt.text}")
            tl = game.time_remaining
            if tl:
                print(f"Time remaining: {tl:.1f}s")

        result = ask_llm(model, question.text, option_texts,
                         competition_id=comp_id, strategy=strategy, use_rag=use_rag,
                         use_tools=use_tools,
                         tools=(tools if tools is not None else maths_tools))
        answer_int = result["answer"]
        if answer_int is None:
            answer_int = max(result["confidence"], key=result["confidence"].get)
            if verbose:
                print(f"Parser failed on {result['raw_output']!r}; "
                      f"fell back to confidence-argmax = {answer_int}")

        chosen_idx = answer_int - 1
        chosen_option_id = question.options[chosen_idx].id

        if verbose:
            print(f"Chose option {answer_int} -> '{option_texts[chosen_idx]}' "
                  f"(latency {result['latency']:.2f}s, "
                  f"conf {result['confidence'][answer_int]:.2f})")

        api_result = game.answer(chosen_option_id)

        if eval_log is not None:
            eval_log.append({
                "strategy": strategy.name,
                "competition": comp_id,
                "level": level,
                "question": question.text,
                "options": option_texts,
                "predicted_idx": chosen_idx,
                "answer": result["answer"],
                "confidence": result["confidence"],
                "raw_output": result["raw_output"],
                "latency": result["latency"],
                "timeout_risk": result["latency"] > LATENCY_DANGER_THRESHOLD_S,
                "correct": bool(api_result.correct),
                "timed_out": bool(api_result.timed_out),
                "earned_after": api_result.earned_amount,
            })

        if question_bank is not None:
            if api_result.correct:
                question_bank.record(
                    question_text=question.text,
                    option_texts=option_texts,
                    correct_idx=chosen_idx,
                    competition_id=comp_id,
                    level=level,
                )
            elif not api_result.timed_out:
                question_bank.eliminate(
                    question_text=question.text,
                    option_texts=option_texts,
                    wrong_idx=chosen_idx,
                    competition_id=comp_id,
                    level=level,
                )

        if api_result.timed_out:
            if verbose:
                print(f"TIMED OUT at level {level}. Final: ${api_result.earned_amount:,.2f}")
            break
        if not api_result.correct:
            if verbose:
                print(f"WRONG. Final: ${api_result.earned_amount:,.2f}")
            break
        if verbose:
            print(f"CORRECT. Earned so far: ${api_result.earned_amount:,.2f}")

    if verbose:
        print(f"\n=== Game over === Reached level {game.current_level}, "
              f"earned ${game.earned_amount:,.2f}")
    return game

# Evaluation (shared)

Drives any approach via `strategy` + `use_tools`,
which covers arms 3/4/5. The sandbox arms (1/2) are evaluated through the arm-aware comparison runners in the
next section (they wrap these same primitives but call the sandbox runner per question).

In [ ]:
# online evaluation function

COMP_NAMES = {
    0: "Entertainment",
    1: "Ancient History and Politics",
    2: "Science and Nature",
    3: "Maths",
}

def online_evaluate(model, client, question_bank=None,
                    comp=3, max_questions=50,
                    strategy=None, use_rag=None,
                    use_tools=False, tools=None):
    """
    Play many games against the live API and log every (question, outcome).
    """
    full_log = []

    with tqdm(total=max_questions, desc=f"Comp {comp}", position=0) as pbar:
        comp_log_count = 0
        while comp_log_count < max_questions:
            game = client.game.start(competition_id=comp)
            start_n = len(full_log)
            play_leaderboard(
                model, game,
                question_bank=question_bank,
                eval_log=full_log,
                verbose=False,
                strategy=strategy,
                use_rag=use_rag,
                use_tools=use_tools,
                tools=tools,
            )
            added = len(full_log) - start_n
            cap = max_questions - comp_log_count
            if added > cap:
                full_log = full_log[: start_n + cap]
                added = cap
            comp_log_count += added
            pbar.update(added)

    df = pd.DataFrame(full_log)
    df["competition_name"] = df["competition"].map(COMP_NAMES)
    return df


def report_results(df, model_name=MODEL_NAME):
    """
    Print accuracy / latency summaries from an evaluation DataFrame.
    """
    print("=" * 64)
    print(f"RESULTS — {model_name}")
    print("=" * 64)

    # Per competition
    print("\nAccuracy per competition (online, biased toward early levels):")
    by_comp = df.groupby("competition_name").agg(
        n=("correct", "size"),
        acc=("correct", "mean"),
        timeouts=("timed_out", "sum"),
    )
    print(by_comp.to_string())

    # Per level (biased — fewer samples at high levels)
    print("\nAccuracy per level (note: smaller n at higher levels):")
    by_level = df.groupby("level").agg(
        n=("correct", "size"),
        acc=("correct", "mean"),
    )
    print(by_level.to_string())

    # Cross
    print("\nAccuracy per competition × level:")
    cross = df.pivot_table(
        index="level", columns="competition_name",
        values="correct", aggfunc="mean",
    )
    print(cross.round(3).to_string())

    # Latency
    print("\nLatency per competition (s):")
    lat_by_comp = df.groupby("competition_name").agg(
        n=("latency", "size"),
        mean_latency=("latency", "mean"),
        max_latency=("latency", "max"),
    )
    print(lat_by_comp.round(3).to_string())
    risky = (df["latency"] > LATENCY_DANGER_THRESHOLD_S).sum()
    print(f"Questions with latency > {LATENCY_DANGER_THRESHOLD_S}s: {risky}/{len(df)}")

# Question Bank (only to populate it, not used in the online game playing)

In [ ]:
class QuestionBank:
    """
    Offline bank of confirmed (question, correct_idx) pairs.
    USED ONLY for offline evaluation. NEVER consulted during play_leaderboard.
    Populated as a side effect when the server confirms an answer correct.
    Key is a stable hash of (question_text, sorted options) so semantically
    identical questions collapse to one entry regardless of option order.
    """

    def __init__(self, path=QUESTION_BANK_PATH):
        self.path = Path(path)
        self.bank = {}
        self.load()

    @staticmethod
    def _key(question_text, option_texts):
        norm_q = question_text.strip().lower()
        norm_opts = "||".join(sorted(o.strip().lower() for o in option_texts))
        h = hashlib.sha256((norm_q + "##" + norm_opts).encode()).hexdigest()
        return h[:16]

    def record(self, question_text, option_texts, correct_idx, competition_id, level):
        """
        Record a confirmed-correct (Q, A).
        """
        k = self._key(question_text, option_texts)
        correct_text = option_texts[correct_idx]
        if k in self.bank:
            entry = self.bank[k]
            entry["seen_count"] += 1
            if entry.get("correct_text") and entry["correct_text"] != correct_text:
                print(f"[bank] WARNING: inconsistent correct_text for {k}")
            entry["correct_text"] = correct_text
            # Any option text that was previously eliminated is now refuted by reality;
            # leave it in eliminated to preserve history, but correct_text wins.
        else:
            self.bank[k] = {
                "question": question_text,
                "options": option_texts,
                "correct_text": correct_text,
                "eliminated_texts": [],
                "competition_id": competition_id,
                "level": level,
                "seen_count": 1,
            }
        self.save()


    def eliminate(self, question_text, option_texts, wrong_idx, competition_id, level):
        """
        Record a confirmed-wrong attempt for a question whose correct answer
        we don't yet know. Builds partial labels over time.
        """
        k = self._key(question_text, option_texts)
        wrong_text = option_texts[wrong_idx]
        if k in self.bank:
            entry = self.bank[k]
            entry["seen_count"] += 1
            if wrong_text not in entry.get("eliminated_texts", []):
                entry.setdefault("eliminated_texts", []).append(wrong_text)
            # If we've now eliminated 3 out of 4 options, the 4th is implicitly correct
            if entry.get("correct_text") is None and len(entry["eliminated_texts"]) == 3:
                remaining = [o for o in option_texts if o not in entry["eliminated_texts"]]
                if len(remaining) == 1:
                    entry["correct_text"] = remaining[0]
        else:
            self.bank[k] = {
                "question": question_text,
                "options": option_texts,
                "correct_text": None,
                "eliminated_texts": [wrong_text],
                "competition_id": competition_id,
                "level": level,
                "seen_count": 1,
            }
        self.save()

    def items_for_eval(self, competition_ids=None, levels=None):
        """
        Return a list of bank entries filtered by competition/level.
        """
        out = []
        for entry in self.bank.values():
            if competition_ids is not None and entry["competition_id"] not in competition_ids:
                continue
            if levels is not None and entry["level"] not in levels:
                continue
            out.append(entry)
        return out

    def load(self):
        if self.path.exists():
            with open(self.path) as f:
                self.bank = json.load(f)

    def save(self):
        self.path.parent.mkdir(parents=True, exist_ok=True)
        with open(self.path, "w") as f:
            json.dump(self.bank, f, indent=2, ensure_ascii=False)

    def stats(self):
        fully_labeled = sum(1 for v in self.bank.values() if v.get("correct_text"))
        partial = sum(1 for v in self.bank.values()
                      if v.get("correct_text") is None and v.get("eliminated_texts"))
        return {
            "total": len(self.bank),
            "fully_labeled": fully_labeled,
            "partially_labeled": partial,
            "unlabeled": len(self.bank) - fully_labeled - partial,
        }

    def __len__(self):
        return len(self.bank)


# Instantiate the bank for this notebook session
question_bank = QuestionBank()
print(f"Loaded question bank with {len(question_bank)} entries")
print(question_bank.stats())

question_bank_eval = QuestionBank(path=QUESTION_BANK_PATH_EVAL)
print(f"Loaded saved question bank for evaluation with {len(question_bank_eval)} entries")
print(question_bank_eval.stats())

# Comparison — ONLINE (live games, competition 3 = Maths)

Plays live games for each arm against the API. Arms 3/4/5 use the shared `online_evaluate`. The sandbox arms
(1/2) use an arm-aware online loop that submits the sandbox runner's pick each level (reusing the same
`play_leaderboard` answer/submit/break logic). Models are loaded once per model. **Latency counts against the
live 30 s timeout** — watch the per-arm latency columns.

In [ ]:
# api retry helpers (from nlp_maths)

def _is_transient_api_error(e):
    """Heuristic: connection errors that may succeed on retry."""
    err_str = str(e).lower()
    return any(token in err_str for token in [
        "connection", "remotedisconnected", "timed out",
        "aborted", "broken pipe", "reset by peer",
    ])


def _retry_api(fn, max_retries=3, base_wait=1.0, label="API call"):
    """Call fn(); on transient errors, exponential-backoff retry.
    Returns the result. Re-raises the last exception if all retries fail
    or the error isn't transient."""
    last_exc = None
    for attempt in range(max_retries):
        try:
            return fn()
        except Exception as e:
            last_exc = e
            if not _is_transient_api_error(e):
                raise
            if attempt < max_retries - 1:
                wait = base_wait * (2 ** attempt)
                print(f"  \u26a0 {label} failed (attempt {attempt + 1}/{max_retries}): {e}")
                print(f"    Retrying in {wait:.1f}s...")
                time.sleep(wait)
    raise last_exc

In [ ]:
import gc

def _free_gpu():
    """Aggressively drop the currently-loaded model from every reference site
    and reclaim VRAM before loading another model."""
    for var in ("model", "tokenizer", "processor", "mistral_tokenizer"):
        if var in globals():
            globals()[var] = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        print(f"  GPU free: {free/1e9:.1f} / {total/1e9:.1f} GB")

In [ ]:
def _online_eval_runner(model, client, runner, label, comp=3, max_questions=50,
                        max_consecutive_failures=3):
    """Online eval for an arbitrary runner (sandbox arms). Plays live games and
    logs rows compatible with report_results.

    Robust to mid-game API drops: game.start / game.answer are wrapped in
    _retry_api (exponential backoff on transient connection errors); a permanent
    failure logs an 'api_error' row and ends the current game gracefully instead
    of crashing the whole comparison."""
    full_log = []
    consecutive_failures = 0
    with tqdm(total=max_questions, desc=f"{label} (comp {comp})", position=0) as pbar:
        n = 0
        while n < max_questions:
            # --- start a game (with retry) ---
            try:
                game = _retry_api(
                    lambda: client.game.start(competition_id=comp),
                    label=f"game.start ({label})",
                )
            except Exception as e:
                consecutive_failures += 1
                print(f"  \u2717 game.start failed: {e}")
                if consecutive_failures >= max_consecutive_failures:
                    print(f"  \u2717 Too many consecutive start failures for {label}; moving on")
                    break
                time.sleep(3.0)
                continue

            start = len(full_log)
            game_broke = False
            try:
                while game.in_progress:
                    question = game.current_question
                    if not question:
                        break
                    option_texts = [o.text for o in question.options]
                    comp_id = game.state.competition.id
                    level = game.current_level
                    res = runner(model, question.text, option_texts, comp_id)
                    ans = res["answer"]
                    if ans is None:
                        ans = max(res["confidence"], key=res["confidence"].get)
                    chosen_idx = ans - 1
                    chosen_option_id = question.options[chosen_idx].id

                    # --- submit answer (with retry) ---
                    api_result = None
                    api_error_msg = None
                    try:
                        api_result = _retry_api(
                            lambda: game.answer(chosen_option_id),
                            label=f"game.answer ({label} L{level})",
                        )
                    except Exception as e:
                        api_error_msg = str(e)
                        print(f"  \u2717 Permanent API failure at L{level}: {e}")

                    full_log.append({
                        "strategy": label,
                        "competition": comp_id,
                        "level": level,
                        "question": question.text,
                        "options": option_texts,
                        "predicted_idx": chosen_idx,
                        "answer": res["answer"],
                        "confidence": res["confidence"],
                        "raw_output": res["raw_output"],
                        "latency": res["latency"],
                        "timeout_risk": res["latency"] > LATENCY_DANGER_THRESHOLD_S,
                        "correct": bool(api_result.correct) if api_result else False,
                        "timed_out": bool(api_result.timed_out) if api_result else False,
                        "earned_after": api_result.earned_amount if api_result else 0.0,
                        "api_error": api_error_msg,
                    })

                    if api_result is None:
                        game_broke = True
                        break
                    if api_result.timed_out or not api_result.correct:
                        break
            except Exception as e:
                consecutive_failures += 1
                print(f"  \u2717 game loop crashed unexpectedly: {e}")
                if consecutive_failures >= max_consecutive_failures:
                    print(f"  \u2717 Too many consecutive crashes for {label}; moving on")
                    break
                time.sleep(3.0)
                continue

            consecutive_failures = 0 if not game_broke else consecutive_failures + 1
            if game_broke and consecutive_failures >= max_consecutive_failures:
                print(f"  \u2717 Too many consecutive API failures for {label}; moving on")
                break

            added = len(full_log) - start
            cap = max_questions - n
            if added > cap:
                full_log = full_log[: start + cap]; added = cap
            n += added
            pbar.update(added)
    df = pd.DataFrame(full_log)
    if not df.empty:
        df["competition_name"] = df["competition"].map(COMP_NAMES)
    return df


_PIPELINE_ONLINE = {
    "3_cot_qwen":      dict(strategy=P_MATHS_COT,  use_tools=False),
    "4_cot_gemma":     dict(strategy=P_MATHS_COT,  use_tools=False),
    "5_baseline_qwen": dict(strategy=P0_BASELINE,  use_tools=False),
}

def compare_arms_online(client, selected=None, comp=3, max_questions=50,
                        question_bank=None):
    """Play live games for the selected arms, swapping models as needed, and
    print a comparison table. A failure inside one arm is caught so the
    remaining arms still run. Returns {label: dataframe}."""
    selected = selected or list(ARMS)
    results = {}
    for mname, arm_list in arms_by_model(selected).items():
        print(f"\n########## loading model: {mname} ##########")
        _free_gpu()
        load_model_globally(mname)
        for label, runner in arm_list:
            print(f"\n--- online arm: {label} ---")
            try:
                if label in _PIPELINE_ONLINE:
                    df = online_evaluate(model, client, question_bank=question_bank,
                                         comp=comp, max_questions=max_questions,
                                         **_PIPELINE_ONLINE[label])
                    if not df.empty:
                        df["strategy"] = label
                else:
                    df = _online_eval_runner(model, client, runner, label,
                                             comp=comp, max_questions=max_questions)
            except Exception as e:
                print(f"  \u2717 arm {label} aborted: {e}")
                df = pd.DataFrame()
            results[label] = df
            if df is not None and not df.empty:
                report_results(df, model_name=label)

    # combined table
    print("\n" + "=" * 86)
    print("ONLINE COMPARISON (Maths, comp %d)" % comp)
    print("=" * 86)
    print(f"{'Arm':<24}{'N':>6}{'Acc':>9}{'timeouts':>10}{'mean s':>10}{'max s':>10}")
    print("-" * 86)
    for label in selected:
        df = results.get(label)
        if df is None or df.empty:
            print(f"{label:<24}{'(no data)':>41}"); continue
        print(f"{label:<24}{len(df):>6}{df['correct'].mean():>9.3f}"
              f"{int(df['timed_out'].sum()):>10}{df['latency'].mean():>10.2f}"
              f"{df['latency'].max():>10.2f}")
    print("=" * 86)
    return results

In [ ]:
# RUN the online comparison against the live API.
# Start small (max_questions) — the sandbox / CoT arms are multi-pass and the 30s wall is real.
online_results = compare_arms_online(
    client,
    selected=["1_sandbox_qwen", "2_sandbox_gated_qwen",
              "3_cot_qwen", "4_cot_gemma", "5_baseline_qwen"],
    comp=3,
    max_questions=500,
    question_bank=question_bank,   # populates the offline bank as a side effect
)


########## loading model: Qwen/Qwen2.5-14B-Instruct ##########
  GPU free: 12.6 / 15.6 GB
Loading Qwen/Qwen2.5-14B-Instruct  (source: hub, path: Qwen/Qwen2.5-14B-Instruct)


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

  loaded: Qwen/Qwen2.5-14B-Instruct

--- online arm: 1_sandbox_qwen ---


1_sandbox_qwen (comp 3):  11%|█         | 55/500 [10:09<2:08:10, 17.28s/it]

  ⚠ game.answer (1_sandbox_qwen L1) failed (attempt 1/3): Could not connect to server at http://131.175.15.22:51111: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
    Retrying in 1.0s...


...

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

  loaded: google/gemma-4-E4B-it

--- online arm: 4_cot_gemma ---


Comp 3:  29%|██▊       | 143/500 [44:41<1:51:34, 18.75s/it]

  ✗ arm 4_cot_gemma aborted: Could not connect to server at http://131.175.15.22:51111: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

ONLINE COMPARISON (Maths, comp 3)
Arm                          N      Acc  timeouts    mean s     max s
--------------------------------------------------------------------------------------
1_sandbox_qwen             500    0.706        51      0.89      1.82
2_sandbox_gated_qwen       500    0.670        27      1.78      2.91
3_cot_qwen                                              (no data)
4_cot_gemma                                             (no data)
5_baseline_qwen            500    0.622         0      0.85      1.43


## Maths — Chain-of-Thought (Qwen2.5-14B), Online

**Overall:** 500 questions, **75.0%** accuracy, 0 timeouts.
Mean latency 7.96s, max 21.91s (11/500 questions over 20s).

### Accuracy per level

| Level | n | Accuracy |
|---:|---:|---:|
| 1 | 128 | 0.727 |
| 2 | 93 | 0.774 |
| 3 | 72 | 0.667 |
| 4 | 48 | 0.854 |
| 5 | 40 | 0.750 |
| 6 | 30 | 0.833 |
| 7 | 25 | 0.800 |
| 8 | 20 | 0.500 |
| 9 | 10 | 1.000 |
| 10 | 10 | 1.000 |
| 11 | 10 | 0.800 |
| 12 | 8 | 0.250 |
| 13 | 2 | 1.000 |
| 14 | 2 | 1.000 |
| 15 | 2 | 1.000 |

*Accuracy is biased toward early levels (larger n); higher levels have few questions, so their rates are noisy.*